# Paper 1 — Golden Age Semantic Reconfiguration

**Current stage: Phase 6 — Temporal Coverage Expansion and Identifiability (historiographic validation integrated)**

Phase 5 established a reproducible baseline of **76 primary-dated poems** (14 A, 62 B), plus 100 Boscán sensitivity-only intervals and 89 Herrera circulation attestations.

Phase 6 asks whether the chronology is sufficiently **author-diverse and historiographically distributed** to identify a Renaissance–Baroque reconfiguration rather than a Garcilaso→Góngora contrast.

The validated auxiliary file `ADSO_processing.xlsx` is used **only as an external literary-historical classification layer** (López Bueno / broader Renaissance–Baroque labels). It is never used to assign poem dates, train a semantic model, or construct network edges.

**Do not build semantic networks yet.**


In [1]:
import re, shutil, subprocess, unicodedata, math
from pathlib import Path
from difflib import SequenceMatcher
import pandas as pd
import xml.etree.ElementTree as ET

SOURCES = {
    "navarro_tei": (
        "https://github.com/bncolorado/CorpusSonetosSigloDeOro.git",
        "092a5fe70a4065a4d84bfed288bffd3851348f9c",
    ),
    "gongora_scholarly": (
        "https://github.com/gongoradigital/gongoraobra.git",
        "3beadeecc059a7cc48499dc2683bb378a2630978",
    ),
    "herrera_stylistics": (
        "https://github.com/lamusadecima/Digital-Stylistics-Applied-to-Golden-Age.git",
        "0de990eac908897b5e931aeb5c496170ccf35bab",
    ),
}

ROOT = Path("/content/gasr_phase6_sources")
ROOT.mkdir(exist_ok=True)

def clone(name, url, commit):
    dst = ROOT / name
    if dst.exists():
        shutil.rmtree(dst)
    subprocess.run(["git", "clone", "--quiet", url, str(dst)], check=True)
    subprocess.run(["git", "-C", str(dst), "checkout", "--quiet", commit], check=True)
    got = subprocess.check_output(
        ["git", "-C", str(dst), "rev-parse", "HEAD"], text=True
    ).strip()
    assert got == commit, (name, got, commit)
    return dst

paths = {k: clone(k, *v) for k, v in SOURCES.items()}
N = paths["navarro_tei"]
G = paths["gongora_scholarly"]
HS = paths["herrera_stylistics"]
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"

def local(tag):
    return tag.split("}")[-1] if "}" in tag else tag

def el_text(el):
    return "" if el is None else " ".join(" ".join(el.itertext()).split())

def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", s.lower())

def years_1580_1626(s):
    return sorted(set(
        int(x) for x in re.findall(r"(?<!\d)(1[56]\d{2})(?!\d)", str(s))
        if 1580 <= int(x) <= 1626
    ))

def segment_blocks(path):
    raw = Path(path).read_text(
        encoding="utf-8", errors="replace"
    ).replace("\r\n", "\n")
    out = []
    for i, block in enumerate(re.split(r"\n\s*\n+", raw), 1):
        lines = [x.strip() for x in block.splitlines() if x.strip()]
        if not lines:
            continue
        txt = "\n".join(lines)
        out.append({
            "block_id": i,
            "n_lines": len(lines),
            "text": txt,
            "signature": norm(txt),
            "first_line": lines[0],
            "first2_signature": norm("\n".join(lines[:2])),
        })
    return pd.DataFrame(out)

def body_title(root):
    for body in root.iter():
        if local(body.tag) == "body":
            for x in body.iter():
                if local(x.tag) == "title":
                    t = el_text(x)
                    if t:
                        return t
            break
    return ""

rows = []
for fp in sorted(N.rglob("*.xml")):
    root = ET.parse(fp).getroot()
    lines = [el_text(x) for x in root.iter() if local(x.tag) == "l"]
    lines = [x for x in lines if x]
    if not lines:
        continue
    author = fp.parent.name
    txt = "\n".join(lines)
    bib = [
        el_text(x) for x in root.iter()
        if local(x.tag) in {"bibl", "witness"} and el_text(x)
    ]
    rows.append({
        "n_id": f"{author}::{fp.name}",
        "author_dir": author,
        "title": body_title(root),
        "n_lines": len(lines),
        "text": txt,
        "signature": norm(txt),
        "first_line": lines[0],
        "first_line_sig": norm(lines[0]),
        "first2_signature": norm("\n".join(lines[:2])),
        "source_bibl": " | ".join(bib[:4]),
        "source_file": str(fp.relative_to(N)),
    })

n = pd.DataFrame(rows)
assert len(n) == 5078, len(n)

priority_A = {
    "GarcilasoDeLaVega", "JuanBoscan", "FernandoDeHerrera", "PedroEspinosa",
    "JuanDeArguijo", "JuanDeJauregui", "LuisCarrilloySotomayor", "Cervantes",
    "Gongora", "LopeDeVega_1", "LopeDeVega_2", "Quevedo",
}

temporal = n[
    ["n_id", "author_dir", "title", "source_file", "first_line", "source_bibl"]
].copy()

for c in [
    "composition_min", "composition_max", "circulation_year",
    "sensitivity_min", "sensitivity_max",
    "composition_not_before", "composition_not_after",
]:
    temporal[c] = pd.NA

for c, v in [
    ("temporal_confidence", "unassigned"),
    ("temporal_basis", ""),
    ("temporal_source", ""),
    ("chronology_status", "undated"),
    ("circulation_basis", ""),
    ("circulation_source", ""),
    ("sensitivity_basis", ""),
    ("sensitivity_source", ""),
    ("sensitivity_status", "none"),
]:
    temporal[c] = v

constraint_log = []

def assign_primary(ids, lo, hi, confidence, basis, source):
    ids = set(ids)
    mask = temporal.n_id.isin(ids)
    if int(mask.sum()) != len(ids):
        missing = ids - set(temporal.loc[mask, "n_id"])
        raise ValueError(f"Missing primary IDs: {sorted(missing)[:5]}")
    for idx in temporal.index[mask]:
        new = (int(lo), int(hi), str(confidence), str(basis), str(source))
        if temporal.at[idx, "chronology_status"] == "undated":
            temporal.at[idx, "composition_min"] = int(lo)
            temporal.at[idx, "composition_max"] = int(hi)
            temporal.at[idx, "temporal_confidence"] = confidence
            temporal.at[idx, "temporal_basis"] = basis
            temporal.at[idx, "temporal_source"] = source
            temporal.at[idx, "chronology_status"] = "primary_dated"
        else:
            old = (
                int(temporal.at[idx, "composition_min"]),
                int(temporal.at[idx, "composition_max"]),
                str(temporal.at[idx, "temporal_confidence"]),
                str(temporal.at[idx, "temporal_basis"]),
                str(temporal.at[idx, "temporal_source"]),
            )
            if old != new:
                raise ValueError(
                    f"Contradictory primary assignment for "
                    f"{temporal.at[idx, 'n_id']}: {old} vs {new}"
                )

def assign_sensitivity(ids, lo, hi, basis, source):
    ids = set(ids)
    mask = temporal.n_id.isin(ids)
    if int(mask.sum()) != len(ids):
        raise ValueError("Missing sensitivity IDs")
    temporal.loc[mask, "sensitivity_min"] = int(lo)
    temporal.loc[mask, "sensitivity_max"] = int(hi)
    temporal.loc[mask, "sensitivity_basis"] = basis
    temporal.loc[mask, "sensitivity_source"] = source
    temporal.loc[mask, "sensitivity_status"] = "sensitivity_only"

def set_circulation(ids, year, basis, source):
    ids = set(ids)
    mask = temporal.n_id.isin(ids)
    if int(mask.sum()) != len(ids):
        raise ValueError("Missing circulation IDs")
    for idx in temporal.index[mask]:
        current = temporal.at[idx, "circulation_year"]
        if pd.isna(current) or int(year) < int(current):
            temporal.at[idx, "circulation_year"] = int(year)
            temporal.at[idx, "circulation_basis"] = basis
            temporal.at[idx, "circulation_source"] = source

def add_constraint(ids, kind, year, confidence, basis, source):
    if kind not in {"not_before", "not_after"}:
        raise ValueError(kind)
    ids = set(ids)
    mask = temporal.n_id.isin(ids)
    if int(mask.sum()) != len(ids):
        raise ValueError("Missing constraint IDs")
    col = "composition_not_before" if kind == "not_before" else "composition_not_after"
    for idx in temporal.index[mask]:
        old = temporal.at[idx, col]
        if pd.isna(old):
            temporal.at[idx, col] = int(year)
        elif kind == "not_before":
            temporal.at[idx, col] = max(int(old), int(year))
        else:
            temporal.at[idx, col] = min(int(old), int(year))
        constraint_log.append({
            "n_id": temporal.at[idx, "n_id"],
            "author_dir": temporal.at[idx, "author_dir"],
            "constraint_kind": kind,
            "constraint_year": int(year),
            "confidence": confidence,
            "basis": basis,
            "source": source,
        })

def resolve_incipit(author, incipit):
    z = n[n.author_dir.eq(author)].copy()
    key = norm(incipit)
    exact = z[z.first_line_sig.eq(key)]
    if len(exact) == 1:
        return exact.iloc[0], "exact_first_line"
    pref = z[
        z.first_line_sig.str.startswith(key)
        | z.first_line_sig.map(lambda x: key.startswith(x))
    ]
    if len(pref) == 1:
        return pref.iloc[0], "unique_normalized_prefix"
    return None, f"unresolved_{len(exact)}eq_{len(pref)}prefix"

print("Pinned sources ready")
print("Navarro poems:", len(n), "| author folders:", n.author_dir.nunique())


Pinned sources ready
Navarro poems: 5078 | author folders: 53


In [2]:
# Reproduce and freeze the corrected Phase-5 baseline.

# --- Góngora scholarly corpus and Phase-4 linkage ---
gfile = G / "gongora_obra-poetica.xml"
groot = ET.parse(gfile).getroot()
parent = {child: par for par in groot.iter() for child in par}
grows = []

for el in groot.iter():
    xid = el.attrib.get(XML_ID, "")
    if local(el.tag) != "div" or not xid.lower().startswith("poem"):
        continue
    lines = [el_text(x) for x in el.iter() if local(x.tag) == "l"]
    lines = [x for x in lines if x]
    if not lines:
        continue

    vals = []
    cur = el
    for _ in range(6):
        vals += list(cur.attrib.values())
        if cur.text:
            vals.append(cur.text)
        for ch in list(cur):
            if local(ch.tag) in {"head", "date", "label"}:
                vals.append(el_text(ch))
            if ch.tail:
                vals.append(ch.tail)
        cur = parent.get(cur)
        if cur is None:
            break

    ys = sorted(set(y for v in vals for y in years_1580_1626(v)))
    txt = "\n".join(lines)
    grows.append({
        "g_id": xid,
        "n_lines": len(lines),
        "text": txt,
        "signature": norm(txt),
        "first_line_sig": norm(lines[0]),
        "first2_signature": norm("\n".join(lines[:2])),
        "scholarly_year": ys[0] if len(ys) == 1 else pd.NA,
        "year_status": (
            "unique" if len(ys) == 1
            else ("ambiguous" if len(ys) > 1 else "missing")
        ),
    })

g = pd.DataFrame(grows)
g14 = g[(g.n_lines == 14) & g.signature.ne("")].copy()
g_by_id = g.set_index("g_id", drop=False)

ng = n[n.author_dir.eq("Gongora")].copy()
sig_to_gids = g14.groupby("signature").g_id.apply(list).to_dict()
links = []

for r in ng.itertuples(index=False):
    exact_ids = sig_to_gids.get(r.signature, [])
    if len(exact_ids) == 1:
        gid, score, method = exact_ids[0], 1.0, "exact"
    else:
        best_gid, best_score = None, -1.0
        for gr in g14.itertuples(index=False):
            sc = SequenceMatcher(None, r.signature, gr.signature).ratio()
            if sc > best_score:
                best_gid, best_score = gr.g_id, sc
        gid, score, method = best_gid, best_score, "fuzzy"
    links.append({
        "n_id": r.n_id,
        "g_id": gid,
        "method": method,
        "score": float(score),
        "preaccept": method == "exact" or score >= 0.98,
    })

glink = pd.DataFrame(links)
pre = glink[glink.preaccept].copy()
collisions = set(pre.g_id.value_counts()[lambda s: s > 1].index)
glink["accept_phase4"] = glink.preaccept & ~glink.g_id.isin(collisions)

acc = glink[glink.accept_phase4].merge(
    g[["g_id", "scholarly_year", "year_status"]],
    on="g_id", how="left",
)
acc = acc[
    acc.year_status.eq("unique") & acc.scholarly_year.notna()
].copy()

for r in acc.itertuples(index=False):
    conf = "A" if r.method == "exact" else "B"
    basis = (
        "scholarly_chronology_year_exact_link"
        if r.method == "exact"
        else "scholarly_chronology_year_fuzzy_link"
    )
    assign_primary(
        [r.n_id], r.scholarly_year, r.scholarly_year, conf, basis,
        "Cátedra Góngora / Carreira-Biblioteca Castro chronology",
    )

# --- Garcilaso conservative seed ---
roman_vals = {"I":1, "V":5, "X":10, "L":50, "C":100, "D":500, "M":1000}

def roman_to_int(s):
    total, prev = 0, 0
    for ch in reversed(str(s).upper()):
        v = roman_vals.get(ch, 0)
        total += -v if v < prev else v
        prev = max(prev, v)
    return total

gar = n[n.author_dir.eq("GarcilasoDeLaVega")].copy()
gar["roman_token"] = gar.title.str.extract(
    r"^\s*-\s*([IVXLCDM]+)\s*-\s*$", expand=False
)
gar["title_no"] = gar.roman_token.map(
    lambda x: roman_to_int(x) if isinstance(x, str) else pd.NA
).astype("Int64")
gar["sonnet_no"] = pd.to_numeric(
    gar.n_id.str.extract(r"_(\d+)\.xml$", expand=False),
    errors="coerce",
).astype("Int64")

assert len(gar) == 38
assert int((gar.title_no == gar.sonnet_no).fillna(False).sum()) == 38

GAR = {
    **{i:(1526,1532,"B","scholarly_phase_interval") for i in [1,2,3,4,6,26,27]},
    25:(1534,1535,"B","scholarly_interval"),
    33:(1535,1535,"A","historically_anchored_scholarly_year"),
    35:(1535,1535,"A","historically_anchored_scholarly_year"),
    **{i:(1533,1535,"B","revised_scholarly_interval") for i in [7,8,12,15,19,28,30,31]},
}

for no, (lo, hi, conf, basis) in GAR.items():
    z = gar[gar.sonnet_no.eq(no)]
    assert len(z) == 1
    assign_primary(
        [z.iloc[0].n_id], lo, hi, conf, basis,
        "Lapesa chronology via E. L. Rivers (CVC) + AISO/AISPI checks",
    )

# --- Góngora Phase-5 variant recovery ---
phase4_nids = set(glink.loc[glink.accept_phase4, "n_id"])
phase4_gids = set(glink.loc[glink.accept_phase4, "g_id"])
unmatched = ng[~ng.n_id.isin(phase4_nids)].copy()
first2_index = g14.groupby("first2_signature").g_id.apply(list).to_dict()
diag = []

for r in unmatched.itertuples(index=False):
    ids2 = first2_index.get(r.first2_signature, [])
    gid = ids2[0] if len(ids2) == 1 else None
    score, year, status = pd.NA, pd.NA, ""
    if gid:
        gr = g_by_id.loc[gid]
        score = SequenceMatcher(None, r.signature, gr.signature).ratio()
        year = gr.scholarly_year
        status = gr.year_status
    diag.append({
        "n_id": r.n_id,
        "first_line": r.first_line,
        "unique_first2_gid": gid,
        "unique_first2_score": score,
        "unique_first2_year": year,
        "year_status": status,
    })

gdiag = pd.DataFrame(diag)
gdiag["preaccept"] = (
    gdiag.unique_first2_gid.notna()
    & pd.to_numeric(gdiag.unique_first2_score, errors="coerce").ge(0.95)
    & gdiag.unique_first2_year.notna()
    & gdiag.year_status.eq("unique")
    & ~gdiag.unique_first2_gid.isin(phase4_gids)
)
new_collisions = set(
    gdiag.loc[gdiag.preaccept, "unique_first2_gid"]
    .value_counts()[lambda s: s > 1].index
)
gdiag["accept_phase5"] = (
    gdiag.preaccept & ~gdiag.unique_first2_gid.isin(new_collisions)
)
new_g = gdiag[gdiag.accept_phase5].copy()

for r in new_g.itertuples(index=False):
    assign_primary(
        [r.n_id],
        r.unique_first2_year, r.unique_first2_year,
        "B", "scholarly_chronology_year_variant_link",
        "Cátedra Góngora; unique first-two-line signature + >=0.95 full-text similarity",
    )

# --- Herrera circulation and Boscán sensitivity ---
U = HS / "corpus" / "untagged_corpus"
H14 = segment_blocks(U / "H.txt")
H14 = H14[H14.n_lines.eq(14)].copy()
P214 = segment_blocks(U / "P2.txt")
P214 = P214[P214.n_lines.eq(14)].copy()

nh = n[n.author_dir.eq("FernandoDeHerrera")].copy()
Hsig = set(H14.signature)
P2sig = set(P214.signature)
h_ids = nh.loc[nh.signature.isin(Hsig), "n_id"].tolist()
p2_ids = nh.loc[
    nh.signature.isin(P2sig) & ~nh.signature.isin(Hsig), "n_id"
].tolist()

set_circulation(
    h_ids, 1582,
    "H / Algunas obras textual layer",
    "Hernández-Lorenzo companion corpus; Algunas obras (1582)",
)
set_circulation(
    p2_ids, 1619,
    "P2 / Versos posthumous textual layer",
    "Hernández-Lorenzo companion corpus; Versos (1619)",
)

nb = n[n.author_dir.eq("JuanBoscan")].copy()
assign_sensitivity(
    nb.n_id.tolist(), 1526, 1542,
    "Boscán Italianate-sonnet activity envelope; sensitivity only",
    "Navagero-Boscán Granada encounter (1526) to Boscán death (1542)",
)

primary5 = temporal[temporal.chronology_status.eq("primary_dated")].copy()
conf5 = primary5.temporal_confidence.value_counts()

assert len(primary5) == 76
assert int(conf5.get("A", 0)) == 14
assert int(conf5.get("B", 0)) == 62
assert int((temporal.sensitivity_status == "sensitivity_only").sum()) == 100
assert int(temporal.circulation_year.notna().sum()) == 89
assert len(new_g) == 5
assert int(
    (
        temporal.author_dir.eq("FernandoDeHerrera")
        & temporal.chronology_status.eq("primary_dated")
    ).sum()
) == 0

print("PHASE-5 BASELINE PRESERVED")
print("  primary=76 | A=14 | B=62 | sensitivity=100 | circulation=89")


PHASE-5 BASELINE PRESERVED
  primary=76 | A=14 | B=62 | sensitivity=100 | circulation=89


## Phase 6A — Historical anchors and one-sided constraints

A printed or manuscript attestation does **not** become an exact composition year. It is represented as a **terminus ante quem** (`composition_not_after`) plus circulation evidence where appropriate.

Historical-event sonnets may enter the primary axis only when the Navarro poem resolves uniquely by normalized incipit and the event provides a defensible chronological anchor.


In [3]:
# One-sided constraints from author death / exact Herrera H layer.
for author, year in {
    "JuanBoscan": 1542,
    "FernandoDeHerrera": 1597,
    "LuisCarrilloySotomayor": 1610,
}.items():
    add_constraint(
        n.loc[n.author_dir.eq(author), "n_id"].tolist(),
        "not_after", year, "C",
        "author_death_upper_bound",
        "Biographical death date; hard upper-bound constraint only",
    )

add_constraint(
    h_ids, "not_after", 1582, "B",
    "exact_H_textual_attestation_TAQ",
    "Algunas obras (1582), exact normalized-text match to H companion layer",
)

PRIMARY_ANCHORS = [
    {
        "author": "Cervantes",
        "incipit": "Vimos en julio otra semana santa",
        "lo": 1596, "hi": 1596, "confidence": "B",
        "basis": "historical_event_sonnet_Cadiz_1596",
        "source": "BNE / Cervantes scholarship: Cádiz episode, 1596",
    },
    {
        "author": "Cervantes",
        "incipit": "Voto a Dios que me espanta esta grandeza",
        "lo": 1598, "hi": 1598, "confidence": "A",
        "basis": "historical_event_sonnet_FelipeII_tomb_1598",
        "source": "Cervantes Virtual: túmulo de Felipe II in Seville, 1598",
    },
    {
        "author": "Cervantes",
        "incipit": "El que subió por sendas nunca usadas",
        "lo": 1597, "hi": 1598, "confidence": "B",
        "basis": "Herrera_death_epitaph_interval",
        "source": "Cervantes identifies poem with Herrera's death; Herrera died 1597",
    },
    {
        "author": "LuisCarrilloySotomayor",
        "incipit": "Así, sagrado mar, nunca te oprima",
        "lo": 1609, "hi": 1609, "confidence": "B",
        "basis": "scholarly_letter_dated_1609",
        "source": "Menéndez Pelayo to Rodríguez Marín, 6 July 1899: sonnet dated 1609",
    },
]

primary_anchor_audit = []
for a in PRIMARY_ANCHORS:
    row, method = resolve_incipit(a["author"], a["incipit"])
    rec = {
        **a,
        "resolution": method,
        "n_id": None,
        "matched_first_line": None,
        "assigned": False,
    }
    if row is not None:
        rec["n_id"] = row.n_id
        rec["matched_first_line"] = row.first_line
        trow = temporal.loc[temporal.n_id.eq(row.n_id)].iloc[0]
        if trow.chronology_status == "undated":
            assign_primary(
                [row.n_id], a["lo"], a["hi"], a["confidence"],
                a["basis"], a["source"],
            )
            rec["assigned"] = True
        else:
            rec["resolution"] += "_already_primary"
    primary_anchor_audit.append(rec)

primary_anchor_audit = pd.DataFrame(primary_anchor_audit)
print("Primary historical-anchor audit")
display(primary_anchor_audit[
    ["author","incipit","resolution","n_id","assigned","lo","hi","confidence"]
])

# Flores de poetas ilustres (1605): candidate incipit links.
# A unique match receives terminus ante quem + circulation, never exact composition
# merely from the publication date.
FLORES_1605 = [
    ("JuanDeArguijo", "Castiga el cielo a Tántalo inhumano", "Arguijo sonnet in Flores"),
    ("JuanDeArguijo", "A quién me quejaré del crudo engaño", "Arguijo sonnet in Flores"),
    ("JuanDeArguijo", "La horrible sima con espanto mira", "Arguijo sonnet in Flores"),
    ("JuanDeArguijo", "Si pudo de Anfión el dulce canto", "Arguijo sonnet in Flores"),
    ("JuanDeArguijo", "Ya el joven fuerte que con muestra hermosa", "Arguijo sonnet in Flores"),
    ("Quevedo", "Estábase la efesia cazadora", "Quevedo first-print attestation"),
    ("Quevedo", "Si con los mismos ojos que leyeres", "Quevedo first-print attestation"),
    ("Quevedo", "La voluntad de Dios por grillos tienes", "Quevedo first-print attestation"),
    ("Quevedo", "Escondido debajo de tu armada", "Quevedo first-print attestation"),
    ("Quevedo", "Mi madre tuve en ásperas montañas", "Quevedo first-print attestation"),
    ("Quevedo", "Sola en ti, Lesbia, vemos ha perdido", "Quevedo first-print attestation"),
    ("Quevedo", "Llegó a los pies de Cristo Madalena", "Quevedo first-print attestation"),
]

flores_audit = []
for author, incipit, note in FLORES_1605:
    row, method = resolve_incipit(author, incipit)
    rec = {
        "author": author,
        "incipit": incipit,
        "resolution": method,
        "n_id": None,
        "matched_first_line": None,
        "constraint_added": False,
        "note": note,
    }
    if row is not None:
        rec["n_id"] = row.n_id
        rec["matched_first_line"] = row.first_line
        add_constraint(
            [row.n_id], "not_after", 1605, "B",
            "Flores_1605_attestation_TAQ",
            "Pedro Espinosa, Primera parte de Flores de poetas ilustres de España (1605)",
        )
        set_circulation(
            [row.n_id], 1605,
            "Flores de poetas ilustres attestation",
            "Pedro Espinosa, Primera parte de Flores de poetas ilustres de España (1605)",
        )
        rec["constraint_added"] = True
    flores_audit.append(rec)

flores_audit = pd.DataFrame(flores_audit)
print("Flores 1605 incipit audit")
display(flores_audit[
    ["author","incipit","resolution","n_id","constraint_added"]
])
print(
    "Resolved Flores constraints:",
    int(flores_audit.constraint_added.sum()), "/", len(flores_audit)
)


Primary historical-anchor audit


,author,incipit,resolution,n_id,assigned,lo,hi,confidence
0,Cervantes,Vimos en julio otra semana santa,exact_first_line,Cervantes::Cervantes_30.xml,True,1596,1596,B
1,Cervantes,Voto a Dios que me espanta esta grandeza,exact_first_line,Cervantes::Cervantes_13.xml,True,1598,1598,A
2,Cervantes,El que subió por sendas nunca usadas,exact_first_line,Cervantes::Cervantes_31.xml,True,1597,1598,B
3,LuisCarrilloySotomayor,"Así, sagrado mar, nunca te oprima",unresolved_0eq_0prefix,None,False,1609,1609,B


Flores 1605 incipit audit


,author,incipit,resolution,n_id,constraint_added
0,JuanDeArguijo,Castiga el cielo a Tántalo inhumano,exact_first_line,JuanDeArguijo::JuanDeArguijo_24.xml,True
1,JuanDeArguijo,A quién me quejaré del crudo engaño,unresolved_0eq_0prefix,None,False
2,JuanDeArguijo,La horrible sima con espanto mira,unresolved_0eq_0prefix,None,False
3,JuanDeArguijo,Si pudo de Anfión el dulce canto,exact_first_line,JuanDeArguijo::JuanDeArguijo_39.xml,True
4,JuanDeArguijo,Ya el joven fuerte que con muestra hermosa,exact_first_line,JuanDeArguijo::JuanDeArguijo_43.xml,True
5,Quevedo,Estábase la efesia cazadora,exact_first_line,Quevedo::Quevedo_318.xml,True
6,Quevedo,Si con los mismos ojos que leyeres,exact_first_line,Quevedo::Quevedo_25.xml,True
7,Quevedo,La voluntad de Dios por grillos tienes,exact_first_line,Quevedo::Quevedo_510.xml,True
8,Quevedo,Escondido debajo de tu armada,exact_first_line,Quevedo::Quevedo_1.xml,True
9,Quevedo,Mi madre tuve en ásperas montañas,exact_first_line,Quevedo::Quevedo_34.xml,True


Resolved Flores constraints: 9 / 12


## Phase 6B — External historiographic classification

The auxiliary spreadsheet recovered from the earlier Hernández-Lorenzo workflow contains author-level literary classifications. For the current Priority-A corpus we retain the López Bueno sequence:

**Innovation → Renovation → Transition → Culmination → Baroque**

This information is treated as an **external validation / stratification variable only**. It does not modify chronology, text representation, semantic similarity, or network construction.

The original local spreadsheet also contains legacy fields such as `Time`, `Time II`, `Birth`, `Death`, and earlier network centralities. Those variables are intentionally excluded from the composition-time reconstruction.


In [4]:
# Auxiliary classification derived from validated local ADSO_processing.xlsx.
# Repository provenance:
# data/derived/auxiliary_literary_classification_priorityA.csv
# The two Navarro Lope folders inherit the same author-level classification.

hist_rows = [
    ("GarcilasoDeLaVega", "Renaissance", "Innovation", "Renaissance", 2),
    ("JuanBoscan", "Renaissance", "Innovation", "Renaissance", 2),
    ("FernandoDeHerrera", "Renaissance", "Renovation", "Renaissance", 1),
    ("JuanDeArguijo", "Baroque", "Transition", "Baroque", 1),
    ("JuanDeJauregui", "Baroque", "Transition", "Baroque", 1),
    ("Cervantes", "Baroque", "Transition", "Baroque", 1),
    ("PedroEspinosa", "Baroque", "Transition", "Baroque", 1),
    ("LuisCarrilloySotomayor", "Baroque", "Transition", "Baroque", 1),
    ("Gongora", "Baroque", "Culmination", "Baroque", 0),
    ("LopeDeVega_1", "Baroque", "Culmination", "Baroque", 0),
    ("LopeDeVega_2", "Baroque", "Culmination", "Baroque", 0),
    ("Quevedo", "Baroque", "Baroque", "Baroque", 0),
]
hist = pd.DataFrame(
    hist_rows,
    columns=[
        "author_dir", "movimiento_literario", "lopez_bueno_2006",
        "pedraza", "legacy_modularity2",
    ],
)
stage_order = {
    "Innovation": 0,
    "Renovation": 1,
    "Transition": 2,
    "Culmination": 3,
    "Baroque": 4,
}
hist["hist_stage_order"] = hist.lopez_bueno_2006.map(stage_order).astype("Int64")
hist["auxiliary_role"] = "external_validation_only"

assert hist.author_dir.is_unique
assert set(hist.author_dir) == priority_A
assert hist.lopez_bueno_2006.notna().all()

temporal = temporal.merge(
    hist[
        ["author_dir","movimiento_literario","lopez_bueno_2006",
         "pedraza","legacy_modularity2","hist_stage_order","auxiliary_role"]
    ],
    on="author_dir", how="left",
)

print("Auxiliary historiographic classification integrated for Priority-A authors.")
display(hist.sort_values(["hist_stage_order","author_dir"]))


Auxiliary historiographic classification integrated for Priority-A authors.


,author_dir,movimiento_literario,lopez_bueno_2006,pedraza,legacy_modularity2,hist_stage_order,auxiliary_role
0,GarcilasoDeLaVega,Renaissance,Innovation,Renaissance,2,0,external_validation_only
1,JuanBoscan,Renaissance,Innovation,Renaissance,2,0,external_validation_only
2,FernandoDeHerrera,Renaissance,Renovation,Renaissance,1,1,external_validation_only
5,Cervantes,Baroque,Transition,Baroque,1,2,external_validation_only
3,JuanDeArguijo,Baroque,Transition,Baroque,1,2,external_validation_only
4,JuanDeJauregui,Baroque,Transition,Baroque,1,2,external_validation_only
7,LuisCarrilloySotomayor,Baroque,Transition,Baroque,1,2,external_validation_only
6,PedroEspinosa,Baroque,Transition,Baroque,1,2,external_validation_only
8,Gongora,Baroque,Culmination,Baroque,0,3,external_validation_only
9,LopeDeVega_1,Baroque,Culmination,Baroque,0,3,external_validation_only


In [5]:
constraints = pd.DataFrame(constraint_log)
if constraints.empty:
    constraints = pd.DataFrame(columns=[
        "n_id","author_dir","constraint_kind","constraint_year",
        "confidence","basis","source",
    ])

# Verify compatibility of primary intervals with one-sided constraints.
primary = temporal[temporal.chronology_status.eq("primary_dated")].copy()
bad = []
for r in primary.itertuples(index=False):
    if (
        pd.notna(r.composition_not_before)
        and int(r.composition_max) < int(r.composition_not_before)
    ):
        bad.append((r.n_id, "not_before"))
    if (
        pd.notna(r.composition_not_after)
        and int(r.composition_min) > int(r.composition_not_after)
    ):
        bad.append((r.n_id, "not_after"))
assert not bad, bad[:10]

def identifiability_class(r):
    if r.chronology_status == "primary_dated":
        return "primary_interval"
    lo = pd.notna(r.composition_not_before)
    hi = pd.notna(r.composition_not_after)
    if lo and hi:
        return "bounded_constraint"
    if lo or hi:
        return "one_sided_constraint"
    if pd.notna(r.circulation_year):
        return "circulation_only"
    if r.sensitivity_status == "sensitivity_only":
        return "sensitivity_only"
    return "unconstrained"

temporal["identifiability_class"] = temporal.apply(
    identifiability_class, axis=1
)

# Author-level coverage.
ident = (
    temporal[temporal.author_dir.isin(priority_A)]
    .groupby(["author_dir","identifiability_class"])
    .size().unstack(fill_value=0)
)
for c in [
    "primary_interval","bounded_constraint","one_sided_constraint",
    "circulation_only","sensitivity_only","unconstrained",
]:
    if c not in ident.columns:
        ident[c] = 0

tot = (
    n[n.author_dir.isin(priority_A)]
    .groupby("author_dir").size().rename("total_poems")
)
ident = ident.join(tot).reset_index().merge(hist, on="author_dir", how="left")
ident["primary_pct"] = (
    100 * ident.primary_interval / ident.total_poems
).round(1)
ident["constraint_or_primary_pct"] = (
    100 * (
        ident.primary_interval
        + ident.bounded_constraint
        + ident.one_sided_constraint
    ) / ident.total_poems
).round(1)
ident = ident.sort_values(
    ["hist_stage_order","primary_pct","constraint_or_primary_pct"],
    ascending=[True,False,False],
)

# Historiographic-stage coverage.
stage_rows = []
for stage, z in temporal[
    temporal.author_dir.isin(priority_A)
].groupby("lopez_bueno_2006", dropna=False):
    author_groups = sorted(z.author_dir.unique())
    p = z[z.identifiability_class.eq("primary_interval")]
    constrained = z[
        z.identifiability_class.isin(
            ["primary_interval","bounded_constraint","one_sided_constraint"]
        )
    ]
    stage_rows.append({
        "lopez_bueno_2006": stage,
        "hist_stage_order": stage_order.get(stage, pd.NA),
        "author_groups": len(author_groups),
        "total_poems": len(z),
        "primary_poems": len(p),
        "primary_author_groups": p.author_dir.nunique(),
        "constraint_or_primary_poems": len(constrained),
        "constraint_or_primary_author_groups": constrained.author_dir.nunique(),
        "circulation_only_poems": int(
            z.identifiability_class.eq("circulation_only").sum()
        ),
        "sensitivity_only_poems": int(
            z.identifiability_class.eq("sensitivity_only").sum()
        ),
        "unconstrained_poems": int(
            z.identifiability_class.eq("unconstrained").sum()
        ),
    })
stage_summary = pd.DataFrame(stage_rows).sort_values("hist_stage_order")

# Dominance / effective-number diagnostics for primary chronology.
primary = temporal[temporal.chronology_status.eq("primary_dated")].copy()
pc = primary.groupby("author_dir").size().sort_values(ascending=False)
p = pc / pc.sum()
primary_effective_authors = float(math.exp(-(p * p.map(math.log)).sum()))
top_author_share = float(p.iloc[0]) if len(p) else 0.0
gongora_share = float(p.get("Gongora", 0.0))
primary_stage_count = int(primary.lopez_bueno_2006.nunique())
primary_author_count = int(primary.author_dir.nunique())

metrics = pd.DataFrame([{
    "primary_poems": len(primary),
    "primary_author_groups": primary_author_count,
    "primary_historiographic_stages": primary_stage_count,
    "effective_primary_authors": round(primary_effective_authors, 3),
    "top_author_share": round(top_author_share, 3),
    "gongora_primary_share": round(gongora_share, 3),
    "constraint_evidence_records": len(constraints),
    "poems_with_one_sided_constraint": int(
        (
            temporal.composition_not_before.notna()
            | temporal.composition_not_after.notna()
        ).sum()
    ),
    "resolved_flores_incipits": int(flores_audit.constraint_added.sum()),
    "primary_anchor_assignments": int(primary_anchor_audit.assigned.sum()),
}])

print("PHASE 6 IDENTIFIABILITY SUMMARY")
display(metrics)
print("\nAuthor-level coverage")
display(ident)
print("\nLópez Bueno stage coverage")
display(stage_summary)

# Priority worklist is driven by historiographic gaps, not raw corpus size.
priority_rank = {
    "Renovation": 1,
    "Transition": 2,
    "Baroque": 3,
    "Innovation": 4,
    "Culmination": 5,
}
work = []
for r in ident.itertuples(index=False):
    if r.primary_interval > 0:
        state = "has_primary"
    elif (r.bounded_constraint + r.one_sided_constraint) > 0:
        state = "constraints_only"
    elif r.circulation_only > 0:
        state = "circulation_only"
    elif r.sensitivity_only > 0:
        state = "sensitivity_only"
    else:
        state = "unconstrained"
    work.append({
        "author_dir": r.author_dir,
        "lopez_bueno_2006": r.lopez_bueno_2006,
        "hist_stage_order": r.hist_stage_order,
        "priority_rank": priority_rank.get(r.lopez_bueno_2006, 9),
        "current_state": state,
        "primary_poems": int(r.primary_interval),
        "constraint_poems": int(r.bounded_constraint + r.one_sided_constraint),
        "circulation_only_poems": int(r.circulation_only),
        "unconstrained_poems": int(r.unconstrained),
        "next_action": (
            "poem-level scholarly chronology / source-group reconstruction"
            if state != "has_primary"
            else "expand only if needed for author balance"
        ),
    })

worklist = pd.DataFrame(work).sort_values(
    ["priority_rank","current_state","author_dir"]
)
print("\nHistoriography-aware chronology worklist")
display(worklist)


PHASE 6 IDENTIFIABILITY SUMMARY


,primary_poems,primary_author_groups,primary_historiographic_stages,effective_primary_authors,top_author_share,gongora_primary_share,constraint_evidence_records,poems_with_one_sided_constraint,resolved_flores_incipits,primary_anchor_assignments
0,79,3,3,1.99,0.734,0.734,505,479,9,3



Author-level coverage


,author_dir,one_sided_constraint,primary_interval,unconstrained,bounded_constraint,circulation_only,sensitivity_only,total_poems,movimiento_literario,lopez_bueno_2006,pedraza,legacy_modularity2,hist_stage_order,auxiliary_role,primary_pct,constraint_or_primary_pct
2,GarcilasoDeLaVega,0,18,20,0,0,0,38,Renaissance,Innovation,Renaissance,2,0,external_validation_only,47.4,47.4
4,JuanBoscan,100,0,0,0,0,0,100,Renaissance,Innovation,Renaissance,2,0,external_validation_only,0.0,100.0
1,FernandoDeHerrera,320,0,0,0,0,0,320,Renaissance,Renovation,Renaissance,1,1,external_validation_only,0.0,100.0
0,Cervantes,0,3,74,0,0,0,77,Baroque,Transition,Baroque,1,2,external_validation_only,3.9,3.9
9,LuisCarrilloySotomayor,50,0,0,0,0,0,50,Baroque,Transition,Baroque,1,2,external_validation_only,0.0,100.0
5,JuanDeArguijo,3,0,67,0,0,0,70,Baroque,Transition,Baroque,1,2,external_validation_only,0.0,4.3
6,JuanDeJauregui,0,0,23,0,0,0,23,Baroque,Transition,Baroque,1,2,external_validation_only,0.0,0.0
10,PedroEspinosa,0,0,20,0,0,0,20,Baroque,Transition,Baroque,1,2,external_validation_only,0.0,0.0
3,Gongora,0,58,57,0,0,0,115,Baroque,Culmination,Baroque,0,3,external_validation_only,50.4,50.4
7,LopeDeVega_1,0,0,699,0,0,0,699,Baroque,Culmination,Baroque,0,3,external_validation_only,0.0,0.0



López Bueno stage coverage


,lopez_bueno_2006,hist_stage_order,author_groups,total_poems,primary_poems,primary_author_groups,constraint_or_primary_poems,constraint_or_primary_author_groups,circulation_only_poems,sensitivity_only_poems,unconstrained_poems
2,Innovation,0,2,138,18,1,118,2,0,0,20
3,Renovation,1,1,320,0,0,320,1,0,0,0
4,Transition,2,5,240,3,1,56,3,0,0,184
1,Culmination,3,3,1461,58,1,58,1,0,0,1403
0,Baroque,4,1,517,0,0,6,1,0,0,511



Historiography-aware chronology worklist


,author_dir,lopez_bueno_2006,hist_stage_order,priority_rank,current_state,primary_poems,constraint_poems,circulation_only_poems,unconstrained_poems,next_action
2,FernandoDeHerrera,Renovation,1,1,constraints_only,0,320,0,0,poem-level scholarly chronology / source-group...
5,JuanDeArguijo,Transition,2,2,constraints_only,0,3,0,67,poem-level scholarly chronology / source-group...
4,LuisCarrilloySotomayor,Transition,2,2,constraints_only,0,50,0,0,poem-level scholarly chronology / source-group...
3,Cervantes,Transition,2,2,has_primary,3,0,0,74,expand only if needed for author balance
6,JuanDeJauregui,Transition,2,2,unconstrained,0,0,0,23,poem-level scholarly chronology / source-group...
7,PedroEspinosa,Transition,2,2,unconstrained,0,0,0,20,poem-level scholarly chronology / source-group...
11,Quevedo,Baroque,4,3,constraints_only,0,6,0,511,poem-level scholarly chronology / source-group...
1,JuanBoscan,Innovation,0,4,constraints_only,0,100,0,0,poem-level scholarly chronology / source-group...
0,GarcilasoDeLaVega,Innovation,0,4,has_primary,18,0,0,20,expand only if needed for author balance
8,Gongora,Culmination,3,5,has_primary,58,0,0,57,expand only if needed for author balance


In [6]:
OUT = Path("/content/gasr_phase6_outputs")
OUT.mkdir(exist_ok=True)

temporal.to_csv(OUT / "temporal_master_phase6.csv", index=False)
constraints.to_csv(OUT / "phase6_constraint_evidence.csv", index=False)
primary_anchor_audit.to_csv(OUT / "phase6_primary_anchor_audit.csv", index=False)
flores_audit.to_csv(OUT / "phase6_flores_incipit_audit.csv", index=False)
hist.to_csv(OUT / "phase6_author_historiography.csv", index=False)
ident.to_csv(OUT / "phase6_identifiability_by_author.csv", index=False)
stage_summary.to_csv(OUT / "phase6_historiographic_stage_coverage.csv", index=False)
metrics.to_csv(OUT / "phase6_identifiability_metrics.csv", index=False)
worklist.to_csv(OUT / "phase6_priority_worklist.csv", index=False)
glink.to_csv(OUT / "gongora_phase4_links.csv", index=False)
gdiag.to_csv(OUT / "gongora_phase5_recovery_diagnostics.csv", index=False)

# Regression checks: Phase-5 evidence must remain intact.
assert len(primary5) == 76
assert int(conf5.get("A", 0)) == 14
assert int(conf5.get("B", 0)) == 62
assert int((temporal.sensitivity_status == "sensitivity_only").sum()) == 100
assert int(
    (
        temporal.author_dir.eq("FernandoDeHerrera")
        & temporal.chronology_status.eq("primary_dated")
    ).sum()
) == 0
assert set(hist.author_dir) == priority_A

print()
print("PHASE 6 CHECKPOINT")
print("------------------")
print("Phase-5 baseline preserved: primary=76; A=14; B=62.")
print("New Phase-6 primary total:", len(primary))
print("Primary author groups:", primary.author_dir.nunique())
print("Primary historiographic stages:", primary.lopez_bueno_2006.nunique())
print("Effective number of primary authors:", round(primary_effective_authors, 3))
print("Top-author primary share:", round(top_author_share, 3))
print("Resolved Flores 1605 incipits:", int(flores_audit.constraint_added.sum()), "/", len(flores_audit))
print("Primary historical anchors assigned:", int(primary_anchor_audit.assigned.sum()), "/", len(primary_anchor_audit))
print("Outputs:", OUT)
print()
print("No semantic windows are selected automatically.")
print("Next decision must use author balance + historiographic-stage coverage.")



PHASE 6 CHECKPOINT
------------------
Phase-5 baseline preserved: primary=76; A=14; B=62.
New Phase-6 primary total: 79
Primary author groups: 3
Primary historiographic stages: 3
Effective number of primary authors: 1.99
Top-author primary share: 0.734
Resolved Flores 1605 incipits: 9 / 12
Primary historical anchors assigned: 3 / 4
Outputs: /content/gasr_phase6_outputs

No semantic windows are selected automatically.
Next decision must use author balance + historiographic-stage coverage.
